In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

# flipped coin 10 times

heads = 7
flips = 10

with pm.Model() as coin_model:
    # i have prior before seeing the data
    theta = pm.Beta("theta", alpha=1, beta=1) # prior, true_bias
    # notion for hidden, true parameter we are hunting, which we ofc have prior about
    #given prior, what is the probab of seeing 7 heads in 10 flips??
    likelihood=pm.Binomial("likelihood", n=flips, p=theta, observed=heads) #y
    # machine running the generative story backward, for every bias/probab
    # if bias was 10%, how many ways could that produce 7 heads in 10?
    # its counting logical paths that lead to the hard reality of our 7 heads


    posterior = pm.sample(1000, tune=1000,chains=2, cores=1, random_seed=42) # trace
    # 1000 draws, cores >=2

    az.plot_trace(posterior)
    plt.show()

    # divergence must be 0, even a 1 divergence means our model is structurally compromised
    # theta peaking around .7

In [ ]:
az.plot_posterior(posterior, var_names=["theta"])
plt.title("Posterior: Probability of Heads")
plt.show()

# 5. THE SUMMARY: Get the numbers
display(az.summary(posterior, var_names=["theta"]))

In [ ]:
fails = 3
tests=10

with pm.Model() as concrete_test:

    # 3 failures out of 10
    # prior -> likelihood (evidence) -> posterior

    prior = pm.Beta("prior", alpha=1, beta=1)
    likelihood = pm.Binomial("likelihood", n=tests, p=prior, observed=fails)
    posterior=pm.sample(1000, tune=1000, cores=2, chains=4, random_seed=42)

    az.plot_trace(posterior)
    plt.show()

    # theta peaking around 0.3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (heads, flips) in enumerate([(7, 10), (70, 100), (700, 1000)]):
    with pm.Model():
        theta = pm.Beta("theta", alpha=1, beta=1)
        y = pm.Binomial("y", n=flips, p=theta, observed=heads)
        trace = pm.sample(1000, tune=1000, cores=1, random_seed=42)

    az.plot_posterior(trace, var_names=["theta"], ax=axes[i])
    axes[i].set_title(f"{heads}/{flips} flips")
    axes[i].set_xlim(0, 1)

plt.tight_layout()
plt.show()


# our hdi shrinks signficantly as our evidence increase

In [ ]:
# strong prior vs sus evidence

sus_trials = 10
sus_heads = 9

with pm.Model() as sus_coin:
    prior = pm.Beta("prior", alpha=50, beta=50)
    likelihood = pm.Binomial("likelihood", n = sus_trials, p = prior, observed = sus_heads)
    posterior=pm.sample(1000, tune=1000, cores=2, chains=4, random_seed=42)

    az.plot_trace(posterior)
    plt.show()

    # posterior is pulled towards prior because of 100 'obs' but data only 10
    # prior around 0.55
    # can have anamolous data to trample on our strong priors

In [ ]:
sus_trials = 100
sus_heads = 90

with pm.Model() as sus_coin:
    prior = pm.Beta("prior", alpha=50, beta=50)
    likelihood = pm.Binomial("likelihood", n = sus_trials, p = prior, observed = sus_heads)
    posterior=pm.sample(1000, tune=1000, cores=2, chains=4, random_seed=42)

    az.plot_trace(posterior)
    plt.show()

    # now the prior shifts because equal (in amt) evidence says otherwise to ~.7